![imagen](./img/python.jpg)

# Extra: archivos XML

En el notebook de lectura/escritura has visto CSV, Excel, JSON, TXT, ZIP y pickle. Hay otro formato de texto plano que te vas a encontrar con bastante frecuencia, sobre todo en sistemas "de toda la vida" (administraciones públicas, bancos, ERPs, feeds RSS, ficheros de configuración...): **XML**.

En este notebook extra verás qué es, cómo leerlo con la librería estándar de Python y cómo convertirlo en un `DataFrame` de `pandas`.

1. [¿Qué es XML?](#1.-¿Qué-es-XML?)
2. [El fichero como texto plano](#2.-El-fichero-como-texto-plano)
3. [ElementTree](#3.-ElementTree)
4. [De XML a DataFrame con ElementTree](#4.-De-XML-a-DataFrame-con-ElementTree)
5. [pd.read_xml()](#5.-pd.read_xml())
6. [XML anidado](#6.-XML-anidado)
7. [Escritura](#7.-Escritura)
8. [¿Cuándo usar cada opción?](#8.-¿Cuándo-usar-cada-opción?)

## 1. ¿Qué es XML?
***eXtensible Markup Language*. Es un lenguaje de marcas, parecido al HTML, pensado para almacenar e intercambiar datos estructurados**. A diferencia del HTML, las etiquetas no están predefinidas: cada fichero define las suyas.

Sus piezas básicas son:

* **Etiqueta (*tag*)**: el nombre del elemento, entre `< >`. Cada etiqueta que se abre (`<TITLE>`) se cierra (`</TITLE>`).
* **Atributo**: pares `clave="valor"` dentro de la etiqueta de apertura, como `id="1"`.
* **Texto**: el contenido que hay entre la apertura y el cierre.

```xml
<?xml version="1.0" encoding="UTF-8"?>   <!-- declaración -->
<CATALOG>                                <!-- nodo raíz -->
  <CD id="1">                            <!-- elemento con un atributo -->
    <TITLE>Empire Burlesque</TITLE>      <!-- elemento con texto -->
    <ARTIST>Bob Dylan</ARTIST>
    ...
  </CD>
  <CD id="2">
    ...
  </CD>
</CATALOG>
```

**Un XML es un árbol**: hay un único nodo raíz (`CATALOG`), que tiene hijos (`CD`), que a su vez tienen hijos (`TITLE`, `ARTIST`...). Esta idea es la clave para leerlo: iremos recorriendo el árbol de padres a hijos.

Si lo comparas con JSON, la idea es la misma (datos jerárquicos en texto plano), pero XML es más verboso y añade los atributos.

## 2. El fichero como texto plano
Al ser texto, podemos abrirlo como cualquier otro fichero para echar un vistazo a su estructura antes de procesarlo.

In [1]:
import os
import pandas as pd

with open('data/cd_catalog.xml', encoding = 'utf-8') as f:
    for linea in f.readlines()[:12]:
        print(linea, end = '')

<?xml version="1.0" encoding="UTF-8"?>
<CATALOG>
  <CD id="1">
    <TITLE>Empire Burlesque</TITLE>
    <ARTIST>Bob Dylan</ARTIST>
    <COUNTRY>USA</COUNTRY>
    <COMPANY>Columbia</COMPANY>
    <PRICE>10.90</PRICE>
    <YEAR>1985</YEAR>
  </CD>
  <CD id="2">
    <TITLE>Hide your heart</TITLE>


## 3. ElementTree
Python trae en su librería estándar el módulo `xml.etree.ElementTree` (no hace falta instalar nada). Por convención se importa como `ET`. [Aquí tienes la documentación](https://docs.python.org/3/library/xml.etree.elementtree.html).

**Lectura**: `ET.parse()` lee el fichero y construye el árbol. Con `getroot()` obtenemos el nodo raíz, desde el que empezaremos a navegar.

In [2]:
import xml.etree.ElementTree as ET

tree = ET.parse('data/cd_catalog.xml')
root = tree.getroot()

print(root.tag)    # nombre de la etiqueta raíz
print(len(root))   # número de hijos directos

CATALOG
26


Cada nodo del árbol es un `Element`, y tiene tres propiedades fundamentales:
* `tag`: el nombre de la etiqueta.
* `attrib`: diccionario con sus atributos.
* `text`: el texto que contiene.

Un `Element` se comporta como una lista de sus hijos, así que podemos indexarlo y recorrerlo.

In [3]:
cd = root[0]   # primer hijo de la raíz

print(cd.tag)
print(cd.attrib)
print(cd.get('id'))   # acceder a un atributo concreto

for hijo in cd:
    print(hijo.tag, '->', hijo.text)

CD
{'id': '1'}
1
TITLE -> Empire Burlesque
ARTIST -> Bob Dylan
COUNTRY -> USA
COMPANY -> Columbia
PRICE -> 10.90
YEAR -> 1985


**Búsqueda de elementos**
* `find('etiqueta')`: devuelve el **primer** hijo con esa etiqueta.
* `findall('etiqueta')`: devuelve **todos** los hijos directos con esa etiqueta.
* `findtext('etiqueta')`: devuelve directamente el texto del primer hijo con esa etiqueta.
* `iter('etiqueta')`: recorre **todo el árbol** (a cualquier profundidad) buscando esa etiqueta.

In [4]:
print(cd.find('TITLE').text)
print(cd.findtext('ARTIST'))
print(len(root.findall('CD')))

Empire Burlesque
Bob Dylan
26


In [5]:
# Todos los artistas, estén al nivel que estén
[artista.text for artista in root.iter('ARTIST')][:5]

['Bob Dylan', 'Bonnie Tyler', 'Dolly Parton', 'Gary Moore', 'Eros Ramazzotti']

## 4. De XML a DataFrame con ElementTree
Lo que normalmente queremos es una tabla: **una fila por cada `CD` y una columna por cada etiqueta hija**. La receta es siempre la misma:

1. Recorrer los elementos que serán filas.
2. Construir un diccionario por fila.
3. Pasar la lista de diccionarios a `pd.DataFrame()`.

In [6]:
filas = []

for cd in root.findall('CD'):
    fila = {'id': int(cd.get('id'))}
    for hijo in cd:
        fila[hijo.tag] = hijo.text
    filas.append(fila)

df_cd = pd.DataFrame(filas).set_index('id')
df_cd.head()

,TITLE,ARTIST,COUNTRY,COMPANY,PRICE,YEAR
id,,,,,,
1,Empire Burlesque,Bob Dylan,USA,Columbia,10.90,1985
2,Hide your heart,Bonnie Tyler,UK,CBS Records,9.90,1988
3,Greatest Hits,Dolly Parton,USA,RCA,9.90,1982
4,Still got the blues,Gary Moore,UK,Virgin records,10.20,1990
5,Eros,Eros Ramazzotti,EU,BMG,9.90,1997


Ojo: **todo lo que viene de un XML es texto**. Hay que convertir los tipos a mano.

In [7]:
df_cd = df_cd.astype({'PRICE': 'float', 'YEAR': 'int'})
df_cd.dtypes

TITLE          str
ARTIST         str
COUNTRY        str
COMPANY        str
PRICE      float64
YEAR         int32
dtype: object

## 5. pd.read_xml()
Si el XML es "plano" (una lista de registros con un solo nivel de hijos), `pandas` lo lee en una línea con `read_xml()`. [Aquí tienes la documentación](https://pandas.pydata.org/docs/reference/api/pandas.read_xml.html).

Fíjate en que los **atributos (`id`) también se convierten en columnas** y en que intenta inferir los tipos.

In [8]:
df_cd2 = pd.read_xml('data/cd_catalog.xml')
df_cd2.head()

,id,TITLE,ARTIST,COUNTRY,COMPANY,PRICE,YEAR
0,1,Empire Burlesque,Bob Dylan,USA,Columbia,10.9,1985
1,2,Hide your heart,Bonnie Tyler,UK,CBS Records,9.9,1988
2,3,Greatest Hits,Dolly Parton,USA,RCA,9.9,1982
3,4,Still got the blues,Gary Moore,UK,Virgin records,10.2,1990
4,5,Eros,Eros Ramazzotti,EU,BMG,9.9,1997


In [9]:
df_cd2.dtypes

id           int64
TITLE          str
ARTIST         str
COUNTRY        str
COMPANY        str
PRICE      float64
YEAR         int64
dtype: object

Algunos argumentos útiles:
* `xpath`: qué nodos se convierten en filas. Por defecto `'./*'` (los hijos de la raíz).
* `dtype`: forzar tipos, igual que en `read_csv()`.
* `parser`: por defecto usa `'lxml'` (hay que tenerlo instalado: `%pip install lxml`). Si no lo tienes, `parser = 'etree'` usa el `ElementTree` de la librería estándar.

In [10]:
pd.read_xml('data/cd_catalog.xml', parser = 'etree', dtype = {'YEAR': 'str'}).dtypes

id           int64
TITLE          str
ARTIST         str
COUNTRY        str
COMPANY        str
PRICE      float64
YEAR           str
dtype: object

## 6. XML anidado
El mundo real no suele ser tan amable. Echa un vistazo a `data/movies.xml`: cada película tiene un `director` y varios `actor`, **cada uno con sus propios hijos**.

In [11]:
with open('data/movies.xml', encoding = 'utf-8') as f:
    print(f.read()[:1500])

<?xml version="1.0" encoding="UTF-8"?>

<movies>
  <movie>
    <title>A History of Violence</title>
    <year>2005</year>
    <country>USA</country>
    <genre>Crime</genre>
    <summary>Tom Stall, a humble family man and owner of a 
	popular neighborhood restaurant, lives a quiet but 
	fulfilling existence in the Midwest. One night Tom 
	foils a crime at his place of business and, to his 
	chagrin, is plastered all over the news for his 
	heroics. Following this, mysterious people follow 
	the Stalls' every move, concerning Tom more than 
	anyone else. As this situation is confronted, more 
	lurks out over where all these occurrences have 
	stemmed from compromising his marriage, family 
	relationship and the main characters' former 
	relations in the process.</summary>
 <director>	    <last_name>Cronenberg</last_name>
	    <first_name>David</first_name>
	    <birth_date>1943</birth_date>
</director> <actor>
	    <first_name>Vigo</first_name>
	    <last_name>Mortensen</last_name>
	   

Si usamos `read_xml()` directamente, solo lee el primer nivel: **`director` y `actor` aparecen vacíos** (solo recoge espacios y saltos de línea), y además solo cabe un actor por película.

In [12]:
df_movies = pd.read_xml('data/movies.xml')
df_movies[['title', 'year', 'director', 'actor']]

,title,year,director,actor
0,A History of Violence,2005,\t,\n\t
1,Heat,1995,\t,\n\t
2,Unforgiven,1992,\t,\n\t
3,Match Point,2005,\t,\n\t
4,Lost in Translation,2003,\t,\n\t
5,Marie Antoinette,2006,\t,\n\t
6,Spider-Man,2002,\t,\n\t


**Opción 1: `xpath`**. Si lo que nos interesa es un nivel concreto, podemos indicarle qué nodos queremos como filas. `'.//actor'` significa "todos los `actor`, a cualquier profundidad".

In [13]:
pd.read_xml('data/movies.xml', xpath = './/actor').head()

,first_name,last_name,birth_date,role
0,Vigo,Mortensen,1958,Tom Stall
1,Maria,Bello,1967,Eddie Stall
2,Ed,Harris,1950,Carl Fogarty
3,William,Hurt,1950,Richie Cusack
4,Al,Pacino,1940,Lt. Vincent Hanna


El problema es que **hemos perdido a qué película pertenece cada actor**. Cuando necesitamos combinar niveles, toca volver a `ElementTree`.

**Opción 2: ElementTree**. Una fila por película, con el nombre del director y los actores concatenados en un string. Aprovechamos para limpiar el `summary`, que viene con saltos de línea y tabuladores.

In [14]:
root_movies = ET.parse('data/movies.xml').getroot()

filas = []
for movie in root_movies.findall('movie'):
    director = movie.find('director')
    actores = [f"{a.findtext('first_name')} {a.findtext('last_name')}" for a in movie.findall('actor')]

    filas.append({
        'title': movie.findtext('title'),
        'year': int(movie.findtext('year')),
        'genre': movie.findtext('genre'),
        'director': f"{director.findtext('first_name')} {director.findtext('last_name')}",
        'actors': ', '.join(actores),
        'summary': ' '.join(movie.findtext('summary', default = '').split())   # alguna película no tiene summary
    })

df_movies = pd.DataFrame(filas)
df_movies

,title,year,genre,director,actors,summary
0,A History of Violence,2005,Crime,David Cronenberg,"Vigo Mortensen, Maria Bello, Ed Harris, Willia...","Tom Stall, a humble family man and owner of a ..."
1,Heat,1995,Crime,Michael Mann,"Al Pacino, Robert De Niro, Val Kilmer, Jon Voight",Hunters and their prey--Neil and his professio...
2,Unforgiven,1992,Western,Clint Eastwood,"Clint Eastwood, Gene Hackman, Morgan Freeman",The town of Big Whisky is full of normal peopl...
3,Match Point,2005,Crime,Woody Allen,"Jonathan Rhys Meyers, Scarlett Johansson","Chris Wilton is a former tennis pro, looking t..."
4,Lost in Translation,2003,Drama,Sofia Coppola,"Scarlett Johansson, Bill Murray",
5,Marie Antoinette,2006,Drama,Sofia Coppola,"Kirsten Dunst, Jason Schwartzman",Based on Antonia Fraser's book about the ill-f...
6,Spider-Man,2002,Action,Sam Raimi,"Kirsten Dunst, Tobey Maguire, Willem Dafoe","On a school field trip, Peter Parker (Maguire)..."


## 7. Escritura
Igual que con `to_csv()` o `to_json()`, tenemos `to_xml()`. [Aquí tienes la documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_xml.html). Con `root_name` y `row_name` elegimos el nombre de la etiqueta raíz y de cada fila.

In [15]:
df_cd.to_xml('data/cd_catalog_out.xml', root_name = 'CATALOG', row_name = 'CD', parser = 'etree')

with open('data/cd_catalog_out.xml', encoding = 'utf-8') as f:
    print(f.read()[:400])

<?xml version="1.0" encoding="utf-8"?>
<CATALOG>
  <CD>
    <id>1</id>
    <TITLE>Empire Burlesque</TITLE>
    <ARTIST>Bob Dylan</ARTIST>
    <COUNTRY>USA</COUNTRY>
    <COMPANY>Columbia</COMPANY>
    <PRICE>10.9</PRICE>
    <YEAR>1985</YEAR>
  </CD>
  <CD>
    <id>2</id>
    <TITLE>Hide your heart</TITLE>
    <ARTIST>Bonnie Tyler</ARTIST>
    <COUNTRY>UK</COUNTRY>
    <COMPANY>CBS Records</COMPAN


## 8. ¿Cuándo usar cada opción?

| | `pd.read_xml()` | `ElementTree` |
|---|---|---|
| **XML plano (un nivel)** | Ideal, una línea | Funciona, pero más código |
| **Extraer un único nivel anidado** | Sí, con `xpath` | Sí |
| **Combinar varios niveles** | No | Sí |
| **Atributos** | Automático como columnas | `attrib` / `get()` |
| **Tipos de datos** | Los infiere | Todo es texto, hay que convertir |

**Regla práctica**: prueba primero con `pd.read_xml()`. Si el resultado tiene columnas vacías o pierde información, recorre el árbol con `ElementTree`.

<table align="left">
 <tr><td width="80"><img src="./img/ejercicio.png" style="width:auto;height:auto"></td>
     <td style="text-align:left">
         <h3>Ejercicio XML</h3>

1. A partir de `data/movies.xml`, crea un `DataFrame` con **una fila por actor** y las columnas `title`, `year`, `first_name`, `last_name` y `role`.
2. Cuenta cuántos actores tiene cada película.

 </td></tr>
</table>

**Limpieza**

Borramos el archivo generado en este notebook para no dejar basura en la carpeta `data`.

In [16]:
if os.path.exists('data/cd_catalog_out.xml'):
    os.remove('data/cd_catalog_out.xml')